# 模型训练与评估



## 1. 学习目标

完成本课程后，你应该能够：

- 理解训练集和测试集的划分
- 掌握常用的评估指标
- 能够使用混淆矩阵分析模型
- 了解模型优化的基本方法
- 能够完整地训练和评估一个文本分类模型

## 2. 为什么需要评估模型？



### 2.1 生活中的例子

想象你在学习开车：

- **练习阶段**：在驾校练习场反复练习（训练）
- **考试阶段**：在考场上测试你的驾驶技能（评估）

如果只在练习场练习，不参加考试，你怎么知道自己真正学会了吗？

**模型评估就是给模型"考试"，看它在新数据上的表现如何。**

### 2.2 评估的重要性

**为什么不能用训练数据来评估？**

这就像用练习题来考试：
- 学生可能只是记住了答案，而不是真正理解
- 模型可能只是记住了训练数据，而不是学会了规律

**这种现象叫做"过拟合"（Overfitting）**：
- 在训练数据上表现很好
- 在新数据上表现很差

## 3. 划分训练集和测试集

### 3.1 什么是训练集和测试集？

**训练集（Training Set）**：
- 用来训练模型
- 模型从这些数据中学习规律

**测试集（Test Set）**：
- 用来评估模型
- 模型从未见过这些数据
- 模拟真实应用场景

**常见划分比例**：
- 训练集：70-80%
- 测试集：20-30%

### 3.2 使用sklearn划分数据集

sklearn提供了train_test_split函数，可以方便地划分数据，将数据集快速分割为 训练集（training set） 和 测试集（test set），核心目的是评估模型在未见过的数据上的泛化能力。

**函数名称：`train_test_split`**

**来自哪个包？**
- `from sklearn.model_selection import train_test_split`
- sklearn是机器学习最常用的Python库

**这个函数做什么？**
- 自动把数据分成训练集和测试集
- 就像洗牌后把一副扑克牌分成两堆

**重要参数说明：**

```
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, 
    test_size=0.2,  # 测试集占20%
    random_state=42  # 随机种子，保证结果可复现
)
```

- texts：原始数据集
- labels：原始标签数据
- test_size：测试集占比，`0.2` 表示20%用于测试 
- random_state：随机种子，设置为固定数字(如42)，每次运行结果相同 

**为什么要设置random_state？**
- 不设置：每次运行，数据划分都不一样
- 设置为42：每次运行，数据划分都一样（方便调试和对比）

**函数的返回值：**
```
X_train  -> 训练文本
X_test   -> 测试文本
y_train  -> 训练标签
y_test   -> 测试标签
```

In [ ]:
from sklearn.model_selection import train_test_split

# 准备数据
texts = [
    "华为 发布 新款 手机",
    "苹果 公司 推出 新 产品",
    "科技 创新 改变 生活",
    "人工智能 技术 发展 迅速",
    "国足 战胜 对手 晋级",
    "篮球 比赛 精彩 激烈",
    "运动员 训练 刻苦 努力",
    "足球 世界杯 开幕"
]

labels = ["科技", "科技", "科技", "科技", "体育", "体育", "体育", "体育"]

# 划分数据集（80%训练，20%测试）
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, 
    test_size=0.2,  # 测试集占20%
    random_state=42  # 随机种子，保证结果可复现
)

print(f"总数据量: {len(texts)}")
print(f"训练集大小: {len(X_train)}")
print(f"测试集大小: {len(X_test)}")
print(f"\n训练集:")
for text, label in zip(X_train, y_train):
    print(f"  [{label}] {text}")
print(f"\n测试集:")
for text, label in zip(X_test, y_test):
    print(f"  [{label}] {text}")

## 4. 评估指标



### 4.1 准确率（Accuracy）

**定义**：分类正确的样本数 / 总样本数

**公式**：
```
准确率 = 正确预测的数量 / 总数量
```

**示例**：
```
测试100条新闻，正确分类90条
准确率 = 90 / 100 = 90%
```

**优点**：直观易懂

**缺点**：在数据不平衡时会误导



**`accuracy_score`函数**

**来自哪个包？**
- `from sklearn.metrics import accuracy_score`

**这个函数做什么？**
- 计算准确率
- 比较真实标签和预测标签有多少是一致的

**参数说明：**
```python
accuracy_score(y_test, y_pred)
# y_test: 真实标签
# y_pred: 预测标签
```

### 4.2 完整的训练和评估流程

导入需要的工具库：文本特征提取、分类模型、准确率评估

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer  # TfidfVectorizer：将文本转换为机器学习能识别的数值特征（TF-IDF算法）
from sklearn.naive_bayes import MultinomialNB # MultinomialNB：多项式朴素贝叶斯分类器（适合文本分类等离散特征场景）
from sklearn.metrics import accuracy_score # accuracy_score：计算模型预测准确率（正确预测数/总预测数）

步骤1：文本特征提取（核心：将文本字符串→数值矩阵，模型无法直接处理文本）

In [ ]:
# 1.1 初始化TF-IDF向量器：创建特征提取工具实例（默认参数，可根据需求调整）
vectorizer = TfidfVectorizer()

# 1.2 对训练集文本做「拟合+转换」：
# fit：学习训练集文本的词汇表、词频等统计信息（比如哪些词出现过、出现频率）
# transform：将训练集每个文本转换为TF-IDF数值向量（每行对应一个文本，每列对应一个词，值为TF-IDF权重）
# 最终X_train_vec是稀疏矩阵，形状为(训练集样本数, 词汇表大小)
X_train_vec = vectorizer.fit_transform(X_train)

# 1.3 对测试集文本做「仅转换」：
# 注意：只能用训练集学习到的词汇表（vectorizer已通过fit记住），避免测试集数据泄露
# 保证训练集和测试集的特征维度、词汇映射完全一致，否则模型无法预测
X_test_vec = vectorizer.transform(X_test)

步骤2：训练分类模型（使用朴素贝叶斯算法，适合文本分类的轻量模型）

In [ ]:
# 2.1 初始化多项式朴素贝叶斯分类器：创建模型实例（默认参数，可调alpha等平滑系数）
classifier = MultinomialNB()

# 2.2 用训练集的数值特征和对应标签训练模型：
# 模型会学习「TF-IDF特征向量 → 标签」的映射规律（比如哪些词的组合更可能属于某类）
classifier.fit(X_train_vec, y_train)

步骤3：用训练好的模型做预测（对测试集进行分类）

In [ ]:
# 输入测试集的TF-IDF特征向量，模型输出每个文本的预测标签
y_pred = classifier.predict(X_test_vec)

步骤4：评估模型性能（计算准确率）

In [ ]:
# 对比测试集的真实标签（y_test）和模型预测标签（y_pred），统计预测正确的比例
accuracy = accuracy_score(y_test, y_pred)

输出结果：格式化显示准确率（保留2位小数的百分比形式）

In [ ]:
print(f"模型准确率: {accuracy:.2%}")
print(f"\n预测结果对比:")
for text, true_label, pred_label in zip(X_test, y_test, y_pred):
    status = "✅" if true_label == pred_label else "❌"
    print(f"{status} 文本: {text}")
    print(f"   真实标签: {true_label}, 预测标签: {pred_label}\n")

### 4.3 精确率、召回率和F1值

当数据不平衡时，准确率可能会误导我们。这时需要更细致的评估指标。

#### 4.3.1 精确率（Precision）

**定义**：在预测为某类别的样本中，真正属于该类别的比例

**公式**：
```
精确率 = 预测正确的数量 / 预测为该类的总数量
```

**生活类比**：
- 你预测10封邮件是垃圾邮件
- 其中8封真的是垃圾邮件
- 精确率 = 8/10 = 80%

**含义**：预测的准不准

#### 4.3.2 召回率（Recall）

**定义**：在真正属于某类别的样本中，被正确预测的比例

**公式**：
```
召回率 = 预测正确的数量 / 真实为该类的总数量
```

**生活类比**：
- 实际有12封垃圾邮件
- 你找出了8封
- 召回率 = 8/12 = 67%

**含义**：找得全不全

#### 4.3.3 F1值（F1-Score）

**定义**：精确率和召回率的调和平均

**公式**：
```
F1 = 2 × (精确率 × 召回率) / (精确率 + 召回率)
```

**含义**：综合考虑精确率和召回率

#### 4.3.4 函数介绍

**1. `precision_score` / `recall_score` / `f1_score`**

**来自哪个包？**
```python
from sklearn.metrics import precision_score, recall_score, f1_score
```

**这些函数做什么？**
- `precision_score`: 计算精确率
- `recall_score`: 计算召回率
- `f1_score`: 计算F1值（精确率和召回率的综合）

核心计算代码   
```
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
```

**2. 重要参数：`average='macro'`**

**什么是average参数？**

当有多个类别时（如"科技"和"体育"），需要指定如何计算总体指标。

**常见的average选项：**

| 选项 | 含义 | 适用场景 |
|------|------|----------|
| `'macro'` | 对每个类别一视同仁，求平均 | 每个类别同样重要 |
| `'weighted'` | 按类别样本数加权平均 | 类别样本数不平衡时 |
| `'micro'` | 把所有样本放在一起计算 | 关注整体表现 |

**举例说明：**

假设有两个类别：
- 科技类：精确率80%，有100个样本
- 体育类：精确率60%，有10个样本

**macro平均：**
- (80% + 60%) / 2 = 70%
- 两个类别权重相同

**weighted平均：**
- (80% × 100 + 60% × 10) / 110 = 77.3%
- 样本多的类别权重大

**💡 本例使用`'macro'`的原因：**
- 我们希望"科技"和"体育"两个类别同等重要
- 即使样本数不同，也公平对待

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# 计算各项指标（使用macro平均，对每个类别一视同仁）
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print("模型评估指标：")
print("="*40)
print(f"准确率 (Accuracy):  {accuracy:.2%}")
print(f"精确率 (Precision): {precision:.2%}")
print(f"召回率 (Recall):    {recall:.2%}")
print(f"F1值 (F1-Score):   {f1:.2%}")

### 4.4 分类报告

sklearn提供了classification_report函数，可以一次性查看所有指标。

**函数名称：`classification_report`**

**来自哪个包？**
```python
from sklearn.metrics import classification_report
```

**这个函数做什么？**
- 一次性生成所有评估指标的详细报告
- 包括每个类别的精确率、召回率、F1值
- 相当于把前面分开计算的指标整合在一起

**输出报告的结构：**

```
              precision    recall  f1-score   support
      类别1       0.85      0.90      0.87        100
      类别2       0.92      0.88      0.90        150
```

**列的含义：**

| 列名 | 含义 | 说明 |
|------|------|------|
| `precision` | 精确率 | 预测为该类的样本中，真正属于该类的比例 |
| `recall` | 召回率 | 真正属于该类的样本中，被正确预测的比例 |
| `f1-score` | F1值 | 精确率和召回率的调和平均 |
| `support` | 支持度 | 该类别在测试集中的真实样本数 |

**底部的汇总行：**

| 行名 | 含义 |
|------|------|
| `accuracy` | 总体准确率 |
| `macro avg` | 宏平均（每个类别权重相同） |
| `weighted avg` | 加权平均（按样本数加权） |

**💡 使用技巧：**
- 快速查看每个类别的表现
- 找出哪个类别表现较差（需要优化）
- support列帮助判断数据是否平衡

In [ ]:
from sklearn.metrics import classification_report

# 生成分类报告
report = classification_report(y_test, y_pred)

print("详细分类报告：")
print("="*60)
print(report)
print("\n指标说明：")
print("- precision: 精确率（预测的准不准）")
print("- recall: 召回率（找得全不全）")
print("- f1-score: F1值（综合指标）")
print("- support: 该类别在测试集中的样本数")

## 5. 混淆矩阵

### 5.1 什么是混淆矩阵？

**混淆矩阵（Confusion Matrix）**：展示模型预测结果和真实标签的对应关系。

**作用**：
- 清楚地看到哪些类别容易被混淆
- 找出模型的薄弱环节
- 指导模型优化方向

**1. `confusion_matrix`**

**来自哪个包？**
```python
from sklearn.metrics import confusion_matrix
```

**这个函数做什么？**
- 生成混淆矩阵
- 返回一个二维数组（表格），显示预测结果和真实标签的对应关系

**返回值格式：**
```python
array([[8, 2],    # 第一个类别的预测情况
       [1, 9]])   # 第二个类别的预测情况
```

---

**2. `classifier.classes_`**

**这是什么？**
- 模型对象的一个属性（不是方法，不需要加括号）
- 返回模型学到的所有类别名称
- 按字母顺序排列

**示例：**
```python
classes = classifier.classes_
# 输出：['体育', '科技']
```

---

**3. 代码中的打印技巧**

**`print(f"{变量:10s}")`的含义：**

| 格式 | 含义 | 示例 |
|------|------|------|
| `:10s` | 字符串占10个字符宽度 | `"科技    "` |
| `:10d` | 整数占10个字符宽度 | `"       5"` |
| `end=""` | 不换行，继续在同一行打印 | 实现表格效果 |

**为什么这样写？**
- 让输出对齐，形成整齐的表格
- 方便阅读混淆矩阵

**💡 理解混淆矩阵的输出：**
```
          科技      体育      (预测)
科技         8         2       <- 真实是科技的样本
体育         1         9       <- 真实是体育的样本
(真实)
```
- 对角线（8和9）：预测正确的数量
- 非对角线（2和1）：预测错误的数量

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

# 计算混淆矩阵
cm = confusion_matrix(y_test, y_pred)

# 获取类别名称
classes = classifier.classes_

print("混淆矩阵：")
print("="*40)
print(f"{'':10s}", end="")
for cls in classes:
    print(f"{cls:10s}", end="")
print("  (预测)")

for i, cls in enumerate(classes):
    print(f"{cls:10s}", end="")
    for j in range(len(classes)):
        print(f"{cm[i][j]:10d}", end="")
    print()
print("(真实)")

### 5.2 混淆矩阵的解读

**对角线上的数字**：预测正确的数量

**非对角线上的数字**：预测错误的数量

**示例解读**：
```
          科技  体育  (预测)
科技       8     2
体育       1     9
(真实)
```

- 8条科技新闻被正确分类为科技
- 2条科技新闻被错误分类为体育
- 1条体育新闻被错误分类为科技
- 9条体育新闻被正确分类为体育

## 6. 模型优化

如果模型效果不理想，我们可以从多个方面进行优化。

### 6.1 优化方向1：改进数据预处理

**可以尝试：**
- 更好的文本清洗
- 添加停用词过滤
- 使用更好的分词工具
- 添加词性标注

**示例**：

In [ ]:
# 示例：添加停用词过滤
stop_words = ['的', '了', '在', '是', '我', '有', '和', '就', '不', '人']

vectorizer_improved = TfidfVectorizer(stop_words=stop_words)
X_train_improved = vectorizer_improved.fit_transform(X_train)
X_test_improved = vectorizer_improved.transform(X_test)

classifier_improved = MultinomialNB()
classifier_improved.fit(X_train_improved, y_train)
y_pred_improved = classifier_improved.predict(X_test_improved)

accuracy_improved = accuracy_score(y_test, y_pred_improved)
print(f"原始准确率: {accuracy:.2%}")
print(f"改进后准确率: {accuracy_improved:.2%}")

### 6.2 优化方向2：调整特征提取方法

**可以尝试：**
- 从词袋模型改为TF-IDF
- 调整n-gram参数（考虑词组）
- 调整最大特征数

**示例**：

In [ ]:
# 示例：使用bigram（考虑两个词的组合）
vectorizer_bigram = TfidfVectorizer(ngram_range=(1, 2))  # 1-gram和2-gram
X_train_bigram = vectorizer_bigram.fit_transform(X_train)
X_test_bigram = vectorizer_bigram.transform(X_test)

classifier_bigram = MultinomialNB()
classifier_bigram.fit(X_train_bigram, y_train)
y_pred_bigram = classifier_bigram.predict(X_test_bigram)

accuracy_bigram = accuracy_score(y_test, y_pred_bigram)
print(f"原始准确率 (1-gram): {accuracy:.2%}")
print(f"改进后准确率 (1-gram + 2-gram): {accuracy_bigram:.2%}")

### 6.3 优化方向3：尝试不同的算法

**可以尝试：**
- 从朴素贝叶斯改为逻辑回归
- 尝试SVM
- 尝试随机森林

**示例**：

In [ ]:
from sklearn.linear_model import LogisticRegression

# 使用逻辑回归
classifier_lr = LogisticRegression(max_iter=1000)
classifier_lr.fit(X_train_vec, y_train)
y_pred_lr = classifier_lr.predict(X_test_vec)

accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"朴素贝叶斯准确率: {accuracy:.2%}")
print(f"逻辑回归准确率: {accuracy_lr:.2%}")

### 6.4 优化方向4：增加训练数据

**最有效的优化方法往往是增加高质量的训练数据。**

**可以尝试：**
- 收集更多数据
- 数据增强（同义词替换、回译等）
- 平衡各类别的数据量

## 7. 课程总结

### 7.1 本节课我们学习了：

**1. 训练集和测试集**
- 训练集用于训练模型
- 测试集用于评估模型
- 常见划分比例：80%训练，20%测试

**2. 评估指标**
- 准确率：最直观的指标
- 精确率：预测的准不准
- 召回率：找得全不全
- F1值：综合指标

**3. 混淆矩阵**
- 展示预测结果和真实标签的对应关系
- 帮助找出模型的薄弱环节

**4. 模型优化**
- 改进数据预处理
- 调整特征提取方法
- 尝试不同的算法
- 增加训练数据

### 7.2 关键要点

- 必须用测试集评估模型，不能用训练集
- 准确率不是唯一指标，要综合考虑
- 混淆矩阵能帮助我们深入理解模型
- 模型优化是一个迭代的过程
- 数据质量比算法选择更重要
